# 03 — Réponses aux questions BGES

Réponses aux 20 questions du projet NF26 — Bilan Gaz à Effet de Serre.

Plage d'analyse : **2026-05-01 → 2026-10-31** (tous les 6 sites).

In [ ]:
import os
from pathlib import Path
import sys
import tomllib

os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

_cfg_file: Path = Path("../config/config.toml")
_project_root: Path = _cfg_file.parent.parent

with _cfg_file.open("rb") as _f:
    _cfg = tomllib.load(_f)

_src_path: Path = (_project_root / _cfg["paths"]["src_path"]).resolve()
_data_path: Path = (_project_root / _cfg["paths"]["data_path"]).resolve()

sys.path.insert(0, str(_src_path))

In [ ]:
from datetime import date

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

from config.settings import ETLConfig
from jobs.daily_load import DailyTables, run_daily_load
from jobs.initial_load import InitialTables, run_initial_load
from utils.spark import get_spark

spark: SparkSession = get_spark()
spark.sparkContext.setLogLevel("WARN")
config: ETLConfig = ETLConfig(data_path=_data_path)

In [ ]:
initial: InitialTables = run_initial_load(spark, config)

DATE_DEBUT: date = date(2026, 5, 1)
DATE_FIN: date = date(2026, 10, 31)

daily: DailyTables = run_daily_load(spark, config, DATE_DEBUT, DATE_FIN, initial)
print("ETL OK")

In [ ]:
# ── Dimensions (initial_load) ──────────────────────────────────────────────
sdf_dim_date: DataFrame = initial.dim_date
sdf_dim_staff: DataFrame = initial.dim_staff
sdf_dim_equipment: DataFrame = initial.dim_equipment
sdf_dim_transport_type: DataFrame = initial.dim_transport_type

# ── Dimensions (daily_load) ────────────────────────────────────────────────
sdf_dim_city: DataFrame = daily.dim_city  # 6 sites + toutes les villes de mission
sdf_dim_trip: DataFrame = daily.dim_trip

# ── Tables de faits (daily_load) ───────────────────────────────────────────
sdf_fact_mission: DataFrame = daily.fact_mission
sdf_fact_equipment: DataFrame = daily.fact_equipment

_tables: dict[str, DataFrame] = {
    "dim_date": sdf_dim_date,
    "dim_staff": sdf_dim_staff,
    "dim_equipment": sdf_dim_equipment,
    "dim_transport_type": sdf_dim_transport_type,
    "dim_city": sdf_dim_city,
    "dim_trip": sdf_dim_trip,
    "fact_mission": sdf_fact_mission,
    "fact_equipment": sdf_fact_equipment,
}

for _name, _sdf in _tables.items():
    _sdf.createOrReplaceTempView(_name)

print(f"{'Table':<25} {'Lignes':>8}  Colonnes")
print("-" * 80)
for _name, _sdf in _tables.items():
    print(f"{_name:<25} {_sdf.count():>8}  {', '.join(_sdf.columns)}")

In [ ]:
for _name, _sdf in _tables.items():
    print(f"\n{'=' * 70}")
    print(_name.upper())
    print("=" * 70)
    _sdf.show(10, truncate=False)

In [ ]:
# Clés primaires des 6 sites org — dérivées depuis DIM_CITY au runtime
_sk_map: dict[str, int] = {
    r["CITY_NAME"]: int(r["SK_CITY"])
    for r in sdf_dim_city.filter(F.col("IS_ORG_SITE"))
    .select("SK_CITY", "CITY_NAME")
    .collect()
}

SK_Paris:    int = _sk_map["Paris"]
SK_London:   int = _sk_map["London"]
SK_NY:       int = _sk_map["New-York"]
SK_LA:       int = _sk_map["Los Angeles"]
SK_Berlin:   int = _sk_map["Berlin"]
SK_Shanghai: int = _sk_map["Shanghai"]

EU_SK:  list[int] = [SK_Berlin, SK_London, SK_Paris]
ORG_SK: list[int] = list(_sk_map.values())

print(_sk_map)

## Section 1 — Premières questions

### Q1 — Combien de cadres travaillent sur le site de Paris ?

In [ ]:
res: int = sdf_dim_staff.filter(
    (F.col("JOB_TITLE") == "Cadre") & (F.col("SK_SITE") == SK_Paris)
).count()
print(f"Q1 — Cadres sur le site de Paris : {res}")

### Q2 — Combien d'ingénieurs Data travaillent sur les sites aux États-Unis ?

In [ ]:
res: int = (
    sdf_dim_staff.filter(F.col("JOB_TITLE") == "Ingénieur Data")
    .join(
        sdf_dim_city.filter(F.col("COUNTRY_ISO2") == "US").select(
            F.col("SK_CITY").alias("SK_SITE")  # renommer pour le join
        ),
        "SK_SITE",
    )
    .count()
)
print(f"Q2 — Ingénieurs Data aux États-Unis : {res}")

### Q3 — Combien d'ingénieurs informaticiens travaillent dans l'organisation (tous sites compris) ?

In [ ]:
res = sdf_dim_staff.filter(F.col("JOB_TITLE") == "Ingénieur Informaticien").count()
print(f"Q3 — Ingénieurs Informaticiens (tous sites) : {res}")

### Q4 — Combien de PC fixes ont été achetés par l'organisation entre juin et septembre 2026 ?

In [ ]:
res: int = (
    sdf_fact_equipment.join(
        sdf_dim_date.filter(F.month(F.col("DATE_ISO")).between(6, 9)).select(
            F.col("SK_DATE").alias("SK_DATE_PURCHASE")
        ),
        "SK_DATE_PURCHASE",
    )
    .join(
        sdf_dim_equipment.filter(F.col("TYPE").like("PC fixe%")).select("SK_EQUIPMENT"),
        "SK_EQUIPMENT",
    )
    .count()
)
print(f"Q4 — PC fixes achetés (juin–septembre 2026) : {res}")

### Q5 — Quelle a été l'impact carbone des PC fixes sans écran entre mai et octobre 2026 ?

In [ ]:
res: float = (
    sdf_fact_equipment
    .join(
        sdf_dim_equipment.filter(F.col("TYPE") == "PC fixe sans ecran").select("SK_EQUIPMENT"),
        "SK_EQUIPMENT",
    )
    .agg(F.sum(F.col("CO2_IMPACT_KG")).alias("TOTAL_CO2"))
    .first()["TOTAL_CO2"] or 0.0
)
print(f"Q5 — Impact CO₂ PC fixes sans écran (mai–oct 2026) : {res / 1000:.3f} tCO₂e")

### Q6 — Quelle a été l'impact carbone des PC portables achetés par les ingénieurs Data entre mai et octobre 2026 sur les sites de Londres et New-York ?

In [ ]:
res: float = (
    sdf_fact_equipment
    .join(
        sdf_dim_equipment.filter(F.col("TYPE") == "PC portable").select("SK_EQUIPMENT"),
        "SK_EQUIPMENT",
    )
    .join(
        sdf_dim_staff
        .filter(
            (F.col("JOB_TITLE") == "Ingénieur Data")
            & F.col("SK_SITE").isin([SK_London, SK_NY])
        )
        .select("SK_STAFF"),
        "SK_STAFF",
    )
    .agg(F.sum(F.col("CO2_IMPACT_KG")).alias("TOTAL_CO2"))
    .first()["TOTAL_CO2"] or 0.0
)
print(f"Q6 — Impact CO₂ PC portables Ingénieurs Data (Londres + NY) : {res / 1000:.3f} tCO₂e")

### Q7 — Quelle a été l'impact carbone des Écrans achetés par les cadres entre juillet et septembre 2026 sur tous les sites de l'organisation ?

In [ ]:
res: float = (
    sdf_fact_equipment
    .join(
        sdf_dim_date
        .filter(F.month(F.col("DATE_ISO")).between(7, 9))
        .select(F.col("SK_DATE").alias("SK_DATE_PURCHASE")),
        "SK_DATE_PURCHASE",
    )
    .join(
        sdf_dim_equipment.filter(F.col("TYPE") == "Ecran").select("SK_EQUIPMENT"),
        "SK_EQUIPMENT",
    )
    .join(
        sdf_dim_staff.filter(F.col("JOB_TITLE") == "Cadre").select("SK_STAFF"),
        "SK_STAFF",
    )
    .agg(F.sum(F.col("CO2_IMPACT_KG")).alias("TOTAL_CO2"))
    .first()["TOTAL_CO2"] or 0.0
)
print(f"Q7 — Impact CO₂ Écrans Cadres (juil–sept 2026) : {res / 1000:.3f} tCO₂e")

### Q8 — Quelle a été l'impact carbone des missions sur les sites Européens entre mai et octobre 2026 ?

In [ ]:
res: float = (
    sdf_fact_mission
    .join(
        sdf_dim_staff.filter(F.col("SK_SITE").isin(EU_SK)).select("SK_STAFF"),
        "SK_STAFF",
    )
    .agg(F.sum(F.col("CO2_IMPACT_KG")).alias("TOTAL_CO2"))
    .first()["TOTAL_CO2"] or 0.0
)
print(f"Q8 — Impact CO₂ missions sites Européens (mai–oct 2026) : {res / 1000:.3f} tCO₂e")

### Q9 — Quels ont été les 5 jours les plus impactants concernant les missions en avion pour les sites Européens de l'organisation ?

In [ ]:
print("Q9 — Top 5 jours (missions en avion, sites EU) :")
(
    sdf_fact_mission
    .join(
        sdf_dim_staff.filter(F.col("SK_SITE").isin(EU_SK)).select("SK_STAFF"),
        "SK_STAFF",
    )
    .join(
        sdf_dim_transport_type.filter(F.col("TRANSPORT_NAME").like("Avion%"))
        .select("SK_TRANSPORT_TYPE"),
        "SK_TRANSPORT_TYPE",
    )
    .join(
        sdf_dim_date.select(F.col("SK_DATE").alias("SK_DATE_MISSION"), "DATE_ISO"),
        "SK_DATE_MISSION",
    )
    .groupBy("DATE_ISO")
    .agg(F.sum(F.col("CO2_IMPACT_KG")).alias("CO2_KG"))
    .orderBy(F.col("CO2_KG").desc())
    .select("DATE_ISO", F.round(F.col("CO2_KG"), 1).alias("CO2_KG"))
    .show(5)
)

### Q10 — Quel a été le secteur d'activité qui a eu le plus d'impact concernant les missions et le matériel informatique sur l'ensemble des sites de l'organisation ?

In [ ]:
sdf_missions_sector: DataFrame = (
    sdf_fact_mission
    .join(sdf_dim_staff.select("SK_STAFF", "ACTIVITY_SECTOR"), "SK_STAFF")
    .groupBy("ACTIVITY_SECTOR")
    .agg(F.sum(F.col("CO2_IMPACT_KG")).alias("CO2_MISSION_KG"))
)
sdf_equip_sector: DataFrame = (
    sdf_fact_equipment
    .join(sdf_dim_staff.select("SK_STAFF", "ACTIVITY_SECTOR"), "SK_STAFF")
    .groupBy("ACTIVITY_SECTOR")
    .agg(F.sum(F.col("CO2_IMPACT_KG")).alias("CO2_EQUIP_KG"))
)
print("Q10 — Impact CO₂ total par secteur d'activité :")
(
    sdf_missions_sector.join(sdf_equip_sector, "ACTIVITY_SECTOR", "outer")
    .withColumn(
        "CO2_TOTAL_KG",
        F.coalesce(F.col("CO2_MISSION_KG"), F.lit(0.0))
        + F.coalesce(F.col("CO2_EQUIP_KG"), F.lit(0.0)),
    )
    .orderBy(F.col("CO2_TOTAL_KG").desc())
    .select(
        "ACTIVITY_SECTOR",
        F.round(F.col("CO2_MISSION_KG"), 1).alias("CO2_MISSION_KG"),
        F.round(F.col("CO2_EQUIP_KG"), 1).alias("CO2_EQUIP_KG"),
        F.round(F.col("CO2_TOTAL_KG"), 1).alias("CO2_TOTAL_KG"),
    )
    .show()
)

### Q11 — Quel site a eu le plus d'impact concernant les missions et le matériel informatique sur l'ensemble des sites de l'organisation ?

In [ ]:
sdf_staff_site: DataFrame = sdf_dim_staff.join(
    sdf_dim_city.select(
        F.col("SK_CITY").alias("SK_SITE"), F.col("CITY_NAME").alias("SITE_NAME")
    ),
    "SK_SITE",
).select("SK_STAFF", "SITE_NAME")

sdf_missions_site: DataFrame = (
    sdf_fact_mission
    .join(sdf_staff_site, "SK_STAFF")
    .groupBy("SITE_NAME")
    .agg(F.sum(F.col("CO2_IMPACT_KG")).alias("CO2_MISSION_KG"))
)
sdf_equip_site: DataFrame = (
    sdf_fact_equipment
    .join(sdf_staff_site, "SK_STAFF")
    .groupBy("SITE_NAME")
    .agg(F.sum(F.col("CO2_IMPACT_KG")).alias("CO2_EQUIP_KG"))
)
print("Q11 — Impact CO₂ total par site :")
(
    sdf_missions_site.join(sdf_equip_site, "SITE_NAME", "outer")
    .withColumn(
        "CO2_TOTAL_KG",
        F.coalesce(F.col("CO2_MISSION_KG"), F.lit(0.0))
        + F.coalesce(F.col("CO2_EQUIP_KG"), F.lit(0.0)),
    )
    .orderBy(F.col("CO2_TOTAL_KG").desc())
    .select(
        "SITE_NAME",
        F.round(F.col("CO2_MISSION_KG"), 1).alias("CO2_MISSION_KG"),
        F.round(F.col("CO2_EQUIP_KG"), 1).alias("CO2_EQUIP_KG"),
        F.round(F.col("CO2_TOTAL_KG"), 1).alias("CO2_TOTAL_KG"),
    )
    .show()
)

### Q12 — Quel a été l'impact carbone des missions reliant chaque site (départ = site org, arrivée = site org) durant le mois de septembre 2026 ?

In [ ]:
res: float = (
    sdf_fact_mission
    .join(
        sdf_dim_date.filter(F.month(F.col("DATE_ISO")) == 9)
        .select(F.col("SK_DATE").alias("SK_DATE_MISSION")),
        "SK_DATE_MISSION",
    )
    .join(
        sdf_dim_trip
        .filter(
            F.col("SK_CITY_ORIGIN").isin(ORG_SK)
            & F.col("SK_CITY_DESTINATION").isin(ORG_SK)
        )
        .select("SK_TRIP"),
        "SK_TRIP",
    )
    .agg(F.sum(F.col("CO2_IMPACT_KG")).alias("TOTAL_CO2"))
    .first()["TOTAL_CO2"] or 0.0
)
print(f"Q12 — Impact CO₂ missions inter-sites (septembre 2026) : {res / 1000:.3f} tCO₂e")

### Q13 — Quel a été l'impact carbone des séminaires en juillet 2026 pour les employés de Los Angeles ?

In [ ]:
# NB : 'Séminaire' n'est pas dans les données (types connus : Conférence,
# Développement, Formation, Rencontre entreprises, Réunion) → résultat = 0
res: float = (
    sdf_fact_mission
    .filter(F.col("MISSION_TYPE") == "Séminaire")
    .join(
        sdf_dim_date.filter(F.month(F.col("DATE_ISO")) == 7)
        .select(F.col("SK_DATE").alias("SK_DATE_MISSION")),
        "SK_DATE_MISSION",
    )
    .join(
        sdf_dim_staff.filter(F.col("SK_SITE") == SK_LA).select("SK_STAFF"),
        "SK_STAFF",
    )
    .agg(F.sum(F.col("CO2_IMPACT_KG")).alias("TOTAL_CO2"))
    .first()["TOTAL_CO2"] or 0.0
)
print(f"Q13 — Impact CO₂ séminaires Los Angeles (juillet 2026) : {res / 1000:.3f} tCO₂e")

### Q14 — Quel secteur d'activité a été le plus impactant pour les missions "conférences" entre mai et septembre 2026 ?

In [ ]:
print("Q14 — Impact CO₂ conférences par secteur d'activité (mai–sept 2026) :")
(
    sdf_fact_mission
    .filter(F.col("MISSION_TYPE") == "Conférence")
    .join(
        sdf_dim_date.filter(F.month(F.col("DATE_ISO")).between(5, 9))
        .select(F.col("SK_DATE").alias("SK_DATE_MISSION")),
        "SK_DATE_MISSION",
    )
    .join(sdf_dim_staff.select("SK_STAFF", "ACTIVITY_SECTOR"), "SK_STAFF")
    .groupBy("ACTIVITY_SECTOR")
    .agg(F.sum(F.col("CO2_IMPACT_KG")).alias("CO2_KG"))
    .orderBy(F.col("CO2_KG").desc())
    .select("ACTIVITY_SECTOR", F.round(F.col("CO2_KG"), 1).alias("CO2_KG"))
    .show()
)

### Q15 — Quel a été l'âge moyen des employés Ingénieurs Data qui sont partis en formations entre juillet et septembre 2026 ?

In [ ]:
_row = (
    sdf_fact_mission
    .filter(F.col("MISSION_TYPE") == "Formation")
    .join(
        sdf_dim_date.filter(F.month(F.col("DATE_ISO")).between(7, 9))
        .select(F.col("SK_DATE").alias("SK_DATE_MISSION"), "DATE_ISO"),
        "SK_DATE_MISSION",
    )
    .join(
        sdf_dim_staff.filter(F.col("JOB_TITLE") == "Ingénieur Data")
        .select("SK_STAFF", "BIRTH_DATE"),
        "SK_STAFF",
    )
    .select(
        F.floor(F.months_between(F.col("DATE_ISO"), F.col("BIRTH_DATE")) / 12).alias("AGE")
    )
    .agg(F.avg(F.col("AGE")).alias("AGE_MOYEN"))
    .first()
)
res: float = float(_row["AGE_MOYEN"]) if _row and _row["AGE_MOYEN"] else 0.0
print(f"Q15 — Âge moyen Ingénieurs Data en formation (juil–sept 2026) : {res:.1f} ans")

### Q16 — Quelle destination a été la plus impactante (en cumul) entre mai et octobre 2026 ?

In [ ]:
print("Q16 — Destinations les plus impactantes (mai–oct 2026) :")
(
    sdf_fact_mission
    .join(sdf_dim_trip.select("SK_TRIP", "SK_CITY_DESTINATION"), "SK_TRIP")
    .join(
        sdf_dim_city.select(
            F.col("SK_CITY").alias("SK_CITY_DESTINATION"),
            F.col("CITY_NAME").alias("DESTINATION"),
        ),
        "SK_CITY_DESTINATION",
    )
    .groupBy("DESTINATION")
    .agg(F.sum(F.col("CO2_IMPACT_KG")).alias("CO2_KG"))
    .orderBy(F.col("CO2_KG").desc())
    .select("DESTINATION", F.round(F.col("CO2_KG"), 1).alias("CO2_KG"))
    .show(5)
)

### Q17 — Quelles ont été les trois catégories de missions les plus impactantes pour les cadres dans les sites Européens en mai 2026 ?

In [ ]:
print("Q17 — Top 3 types de missions (Cadres, sites EU, mai 2026) :")
(
    sdf_fact_mission
    .join(
        sdf_dim_date.filter(F.month(F.col("DATE_ISO")) == 5)
        .select(F.col("SK_DATE").alias("SK_DATE_MISSION")),
        "SK_DATE_MISSION",
    )
    .join(
        sdf_dim_staff
        .filter((F.col("JOB_TITLE") == "Cadre") & F.col("SK_SITE").isin(EU_SK))
        .select("SK_STAFF"),
        "SK_STAFF",
    )
    .groupBy("MISSION_TYPE")
    .agg(F.sum(F.col("CO2_IMPACT_KG")).alias("CO2_KG"))
    .orderBy(F.col("CO2_KG").desc())
    .select("MISSION_TYPE", F.round(F.col("CO2_KG"), 1).alias("CO2_KG"))
    .show(3)
)

## Section 2 — Questions avec illustrations

### Q18 — Quelles ont été les 5 missions les plus impactantes sur le site de Paris ?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

print("Q18 — Top 5 missions les plus impactantes (Paris) :")
pdf_top5: pd.DataFrame = (
    sdf_fact_mission
    .join(sdf_dim_staff.filter(F.col("SK_SITE") == SK_Paris).select("SK_STAFF"), "SK_STAFF")
    .join(sdf_dim_transport_type.select("SK_TRANSPORT_TYPE", "TRANSPORT_NAME"), "SK_TRANSPORT_TYPE")
    .orderBy(F.col("CO2_IMPACT_KG").desc())
    .limit(5)
    .select(
        "NK_MISSION", "MISSION_TYPE", "TRANSPORT_NAME",
        F.round(F.col("CO2_IMPACT_KG"), 1).alias("CO2_KG"),
    )
    .toPandas()
)
print(pdf_top5.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(pdf_top5["NK_MISSION"], pdf_top5["CO2_KG"])
ax.set_xlabel("CO₂ (kg)")
ax.set_title("Q18 — 5 missions les plus impactantes — Paris")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### Q19 — Figure comparant l'impact carbone mensuel des missions en fonction du type de transport et sur chaque site

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

pdf_q19: pd.DataFrame = (
    sdf_fact_mission
    .join(
        sdf_dim_staff.join(
            sdf_dim_city.select(
                F.col("SK_CITY").alias("SK_SITE"), F.col("CITY_NAME").alias("SITE_NAME")
            ),
            "SK_SITE",
        ).select("SK_STAFF", "SITE_NAME"),
        "SK_STAFF",
    )
    .join(sdf_dim_transport_type.select("SK_TRANSPORT_TYPE", "TRANSPORT_NAME"), "SK_TRANSPORT_TYPE")
    .join(
        sdf_dim_date.select(F.col("SK_DATE").alias("SK_DATE_MISSION"), "DATE_ISO"),
        "SK_DATE_MISSION",
    )
    .groupBy("SITE_NAME", F.month(F.col("DATE_ISO")).alias("MONTH"), "TRANSPORT_NAME")
    .agg(F.round(F.sum(F.col("CO2_IMPACT_KG")) / 1000, 2).alias("tCO2e"))
    .orderBy("SITE_NAME", "MONTH")
    .toPandas()
)

sites: list[str] = sorted(pdf_q19["SITE_NAME"].unique())
months: list[int] = sorted(pdf_q19["MONTH"].unique())

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharey=False)
axes_flat = axes.flatten()

for ax, site in zip(axes_flat, sites):
    pdf_site: pd.DataFrame = pdf_q19[pdf_q19["SITE_NAME"] == site]
    pivot: pd.DataFrame = (
        pdf_site.pivot_table(
            index="MONTH", columns="TRANSPORT_NAME", values="tCO2e", aggfunc="sum"
        )
        .reindex(months)
        .fillna(0)
    )
    pivot.plot(kind="bar", ax=ax, legend=False)
    ax.set_title(site)
    ax.set_xlabel("Mois")
    ax.set_ylabel("tCO₂e")
    ax.set_xticklabels([str(m) for m in months], rotation=0)

handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, title="Transport")
fig.suptitle("Q19 — Impact carbone mensuel des missions par transport et par site", fontsize=13)
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

### Q20 — Figure illustrant l'impact carbone global mensuel de l'organisation

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

pdf_miss: pd.DataFrame = (
    sdf_fact_mission
    .join(sdf_dim_date.select(F.col("SK_DATE").alias("SK_DATE_MISSION"), "DATE_ISO"), "SK_DATE_MISSION")
    .groupBy(F.month(F.col("DATE_ISO")).alias("MONTH"))
    .agg(F.round(F.sum(F.col("CO2_IMPACT_KG")) / 1000, 2).alias("tCO2e_missions"))
    .toPandas()
)
pdf_equip: pd.DataFrame = (
    sdf_fact_equipment
    .join(sdf_dim_date.select(F.col("SK_DATE").alias("SK_DATE_PURCHASE"), "DATE_ISO"), "SK_DATE_PURCHASE")
    .groupBy(F.month(F.col("DATE_ISO")).alias("MONTH"))
    .agg(F.round(F.sum(F.col("CO2_IMPACT_KG")) / 1000, 2).alias("tCO2e_equipements"))
    .toPandas()
)
pdf_q20: pd.DataFrame = (
    pdf_miss.merge(pdf_equip, on="MONTH", how="outer")
    .fillna(0)
    .sort_values("MONTH")
    .reset_index(drop=True)
)
pdf_q20["tCO2e_total"] = pdf_q20["tCO2e_missions"] + pdf_q20["tCO2e_equipements"]

fig, ax = plt.subplots(figsize=(10, 5))
x = pdf_q20["MONTH"]
ax.bar(x - 0.2, pdf_q20["tCO2e_missions"], width=0.35, label="Missions")
ax.bar(x + 0.15, pdf_q20["tCO2e_equipements"], width=0.35, label="Équipements")
ax.plot(x, pdf_q20["tCO2e_total"], marker="o", color="black", label="Total")
ax.set_xlabel("Mois")
ax.set_ylabel("tCO₂e")
ax.set_title("Q20 — Impact carbone global mensuel de l'organisation (mai–oct 2026)")
ax.set_xticks(x)
ax.legend()
plt.tight_layout()
plt.show()